In [ ]:
import pandas as pd
import os
from openpyxl import load_workbook

# Define years to process (change if your scenario uses other years)

years = [2025, 2030, 2035, 2040, 2045, 2050]

# Define folders that contains the OPERA (ESM) output files (put you own path)
opera_folder ='/Users/ahmedelberry/Library/CloudStorage/OneDrive-UvA/GEM-E3/Linking/OPERA results/Base 6'
# two template files: one for energy and one for investments
energy_output = '/Users/ahmedelberry/Library/CloudStorage/OneDrive-UvA/GEM-E3/Linking/Book-EnergyBal with biofuels.xlsx'
cost_output = '/Users/ahmedelberry/Library/CloudStorage/OneDrive-UvA/Paper 3/Inv-python.xlsx'

# remove old data from a sheet (keep headers only)
def clear_sheet_from_row(filepath, sheetname, start_row=2):
    wb = load_workbook(filepath)
    if sheetname in wb.sheetnames:
        ws = wb[sheetname]
        max_row = ws.max_row
        # clear old values below the header row
        for row in range(start_row, max_row + 1):
            for col in range(1, ws.max_column + 1):
                ws.cell(row=row, column=col).value = None
        wb.save(filepath)

# read the OPERA energy balance (technology – carrier – value)        
def read_energy_sheet_precisely(filepath):
    wb = load_workbook(filepath, data_only=True)
    ws = wb["Energy balance options"]
    data = []
     # OPERA: useful data starts at row 6 (change in accordance to your files)
    for row in ws.iter_rows(min_row=6, values_only=True):
        data.append([row[0], row[1], row[3]])  #A: technology, B: carrier, D: input value
    return pd.DataFrame(data, columns=["Col_A", "Col_B", "Col_D"])

# read the OPERA cost sheet (technology – total investment)
def read_cost_sheet_precisely(filepath):
    wb = load_workbook(filepath, data_only=True)
    ws = wb["Cost per option"]
    data = []
    # OPERA: useful data starts at row 11 (change in accordance to your files)
    for row in ws.iter_rows(min_row=11, values_only=True):
        data.append([row[0], row[6]])  # A: technology, G: investment total
    return pd.DataFrame(data, columns=["Col_A", "Col_G"])

# PROCESS_main loop over all years
for year in years:
     # OPERA file for this year (change the name before {year} to be the same as you excel output files names
    opera_file = os.path.join(opera_folder, f"Results Opera SUS NL TRANSFORM {year}.xlsx")

    # --- ENERGY BALANCE ---
    try:
        energy_df = read_energy_sheet_precisely(opera_file)
        # clear the template sheet for this year
        clear_sheet_from_row(energy_output, str(year), start_row=2)
        
        # write the extracted energy data (no header, start below header row)
        with pd.ExcelWriter(energy_output, engine='openpyxl', mode='a', if_sheet_exists='overlay') as writer:
            energy_df.to_excel(writer, sheet_name=str(year), index=False, header=False, startrow=1)

        print(f"[✓] Energy data written for {year}")
    except Exception as e:
        print(f"[!] Failed writing energy for {year}: {e}")

    # --- COST PER OPTION ---
    try:
        cost_df = read_cost_sheet_precisely(opera_file)
         # clear the investment template for this year
        clear_sheet_from_row(cost_output, str(year), start_row=2)
        
        # write extracted investments
        with pd.ExcelWriter(cost_output, engine='openpyxl', mode='a', if_sheet_exists='overlay') as writer:
            cost_df.to_excel(writer, sheet_name=str(year), index=False, header=False, startrow=1)

        print(f"[✓] Investment data written for {year}")
    except Exception as e:
        print(f"[!] Failed writing cost for {year}: {e}")


[✓] Energy data written for 2025
[✓] Investment data written for 2025
[✓] Energy data written for 2030
[✓] Investment data written for 2030
[✓] Energy data written for 2035
[✓] Investment data written for 2035
[✓] Energy data written for 2040
[✓] Investment data written for 2040
[✓] Energy data written for 2045
[✓] Investment data written for 2045
[✓] Energy data written for 2050
